# MST-GNN Experiment B — CSI500 + VN100

**Multilayer Spatial-Temporal Graph Neural Network for Stock Prediction**

| Dataset | Stocks | Period | Est. Runtime |
|---------|--------|--------|--------------|
| CSI500  | ~500   | 2018-01-02 → 2026-06-26 | ~3h |
| VN100   | ~100   | 2020-01-02 → 2026-06-26 | ~2h |

**Features:**
- ✅ Graph cache (pickle) — skip rebuild on re-run
- ✅ Periodic checkpoint (every 20 epochs) — resume on crash
- ✅ Auto-push results to GitHub
- ✅ Early stopping (patience=20)
- ✅ Full metrics + charts after each experiment

> **Note:** Run Notebook A (CSI300 + VN30) first to validate the pipeline.

---
## Step 1 — Environment Check

In [ ]:
import torch, subprocess, os, sys, time, json, glob

print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be very slow!")

assert torch.cuda.is_available(), "GPU required for this notebook!"

---
## Step 2 — Clone / Pull Repository

In [ ]:
REPO_URL = "https://github.com/quocnguyen5/mst-gnn-impl.git"
WORK_DIR = "/kaggle/working/mst-gnn-impl"

if os.path.exists(WORK_DIR):
    print("Repo exists — pulling latest...")
    !cd {WORK_DIR} && git pull
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {WORK_DIR}

os.chdir(WORK_DIR)
print(f"\nWorking dir: {os.getcwd()}")
!git log --oneline -5

---
## Step 3 — Install Dependencies + GitHub Token

In [ ]:
!pip install -q akshare vnstock jieba pyarrow fastparquet tqdm 2>&1 | tail -5

# Setup GitHub token for auto-push
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    os.environ["GITHUB_TOKEN"] = token
    !git remote set-url origin https://{token}@github.com/quocnguyen5/mst-gnn-impl.git
    !git config user.email "nguyennq.app@gmail.com"
    !git config user.name "Kaggle AutoPush"
    print("✅ GitHub token configured — auto-push enabled.")
except Exception as e:
    print(f"⚠️  GitHub token not available ({e}). Auto-push disabled.")
    os.environ["GITHUB_TOKEN"] = ""

print("\n✅ Dependencies installed.")

---
## Step 4 — Run CSI500 Experiment (2018-2026)

Estimated: **~3 hours** on T4 GPU
- Phase 3 (graph build): ~50 min (cached on re-run)
- Phase 5 (training): ~120 min (resumes from checkpoint on crash)

In [ ]:
t_start = time.time()

proc = subprocess.Popen(
    [sys.executable, "-m", "experiments.run_main",
     "--dataset", "csi500", "--aggregator", "mean"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    cwd=WORK_DIR,
    env={**os.environ},
)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.wait()

elapsed = (time.time() - t_start) / 60
status = "✅ SUCCESS" if proc.returncode == 0 else f"❌ FAILED (code {proc.returncode})"
print(f"\nCSI500 {status} — {elapsed:.1f} min")

---
## Step 4b — CSI500 Results

### 4b.1 Test Metrics

In [ ]:
import re, pandas as pd
from IPython.display import display, HTML

def extract_metrics_from_logs(log_dir="logs"):
    """Extract epoch-by-epoch metrics from log files."""
    log_files = sorted(glob.glob(os.path.join(log_dir, "*.log")))
    if not log_files:
        return [], []
    
    train_metrics, val_metrics = [], []
    pattern = re.compile(
        r"Epoch\s+(\d+)/\d+.*?"
        r"Train Loss:\s+([\d.]+).*?Acc:\s+([\d.]+).*?DAMRR:\s+([\d.]+).*?"
        r"Val Loss:\s+([\d.]+).*?Acc:\s+([\d.]+).*?DAMRR:\s+([\d.]+)"
    )
    
    for log_file in log_files:
        with open(log_file) as f:
            for line in f:
                m = pattern.search(line)
                if m:
                    epoch = int(m.group(1))
                    train_metrics.append({
                        'epoch': epoch,
                        'loss': float(m.group(2)),
                        'accuracy': float(m.group(3)),
                        'damrr': float(m.group(4)),
                    })
                    val_metrics.append({
                        'epoch': epoch,
                        'loss': float(m.group(5)),
                        'accuracy': float(m.group(6)),
                        'damrr': float(m.group(7)),
                    })
    return train_metrics, val_metrics

# Display CSI500 test results
print("═" * 50)
print("  CSI500 TEST RESULTS")
print("═" * 50)

ckpt_path = "checkpoints/best_model.pt"
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = ckpt.get("metrics", {})
    print(f"  Best Epoch:  {ckpt.get('epoch', '?')}")
    for k, v in m.items():
        print(f"  {k:12s}: {v:.4f}")
else:
    print("  (checkpoint not found — check log output above)")

### 4b.2 Learning Curves

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def plot_learning_curves(train_metrics, val_metrics, title="", save_path=None):
    """Plot train/val loss and accuracy curves."""
    if not train_metrics:
        print("No metrics to plot.")
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    epochs = [m['epoch'] for m in train_metrics]
    
    # Loss
    axes[0].plot(epochs, [m['loss'] for m in train_metrics], 'b-', label='Train', alpha=0.8)
    axes[0].plot(epochs, [m['loss'] for m in val_metrics], 'r-', label='Val', alpha=0.8)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(epochs, [m['accuracy'] for m in train_metrics], 'b-', label='Train', alpha=0.8)
    axes[1].plot(epochs, [m['accuracy'] for m in val_metrics], 'r-', label='Val', alpha=0.8)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # DAMRR
    axes[2].plot(epochs, [m['damrr'] for m in train_metrics], 'b-', label='Train', alpha=0.8)
    axes[2].plot(epochs, [m['damrr'] for m in val_metrics], 'r-', label='Val', alpha=0.8)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('DAMRR')
    axes[2].set_title('DAMRR')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# Plot CSI500 learning curves
train_m, val_m = extract_metrics_from_logs("logs")
plot_learning_curves(train_m, val_m,
                     title="CSI500 Learning Curves (2018-2026)",
                     save_path="checkpoints/csi500_learning_curves.png")

### 4b.3 Backtest — Cumulative Returns

In [ ]:
from IPython.display import Image as IPImage

chart_path = "checkpoints/cumulative_returns_csi500.png"
if os.path.exists(chart_path):
    display(IPImage(filename=chart_path))
else:
    print(f"Chart not found: {chart_path}")
    print("This is generated during Phase 6 of the experiment.")

---
## Step 5 — Run VN100 Experiment (2020-2026)

Estimated: **~2 hours** on T4 GPU

In [ ]:
t_start = time.time()

proc = subprocess.Popen(
    [sys.executable, "-m", "experiments.run_vn",
     "--universe", "vn100", "--aggregator", "mean",
     "--start", "2020-01-02", "--end", "2026-06-26"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    cwd=WORK_DIR,
    env={**os.environ},
)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.wait()

elapsed = (time.time() - t_start) / 60
status = "✅ SUCCESS" if proc.returncode == 0 else f"❌ FAILED (code {proc.returncode})"
print(f"\nVN100 {status} — {elapsed:.1f} min")

---
## Step 5b — VN100 Results

### 5b.1 Test Metrics

In [ ]:
print("═" * 50)
print("  VN100 TEST RESULTS")
print("═" * 50)

ckpt_path = "checkpoints/best_model.pt"
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = ckpt.get("metrics", {})
    print(f"  Best Epoch:  {ckpt.get('epoch', '?')}")
    for k, v in m.items():
        print(f"  {k:12s}: {v:.4f}")
else:
    print("  (checkpoint not found)")

### 5b.2 Learning Curves

In [ ]:
train_m, val_m = extract_metrics_from_logs("logs")
plot_learning_curves(train_m, val_m,
                     title="VN100 Learning Curves (2020-2026)",
                     save_path="checkpoints/vn100_learning_curves.png")

### 5b.3 Backtest — Cumulative Returns

In [ ]:
chart_path = "checkpoints/cumulative_returns_vn100.png"
if os.path.exists(chart_path):
    display(IPImage(filename=chart_path))
else:
    print(f"Chart not found: {chart_path}")

---
## Step 6 — Final Comparison: CSI500 vs VN100

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

def extract_final_results(log_dir="logs"):
    """Extract final test results from all log files."""
    results = {}
    log_files = sorted(glob.glob(os.path.join(log_dir, "*.log")))
    
    for log_file in log_files:
        with open(log_file) as f:
            content = f.read()
        
        dataset_match = re.search(r"(CSI300|CSI500|VN30|VN100)", content, re.IGNORECASE)
        if not dataset_match:
            continue
        dataset = dataset_match.group(1).upper()
        
        acc = re.findall(r"Test.*?Accuracy:\s+([\d.]+)", content)
        prec = re.findall(r"Test.*?Precision:\s+([\d.]+)", content)
        damrr = re.findall(r"Test.*?DAMRR:\s+([\d.]+)", content)
        
        if acc and prec and damrr:
            results[dataset] = {
                'Accuracy': float(acc[-1]),
                'Precision': float(prec[-1]),
                'DAMRR': float(damrr[-1]),
            }
    
    return results

results = extract_final_results("logs")

if results:
    df = pd.DataFrame(results).T
    df.index.name = 'Dataset'
    
    print("\n" + "═" * 60)
    print("  FINAL COMPARISON — MST-GNN (Mean Aggregator)")
    print("═" * 60)
    display(df.style.format("{:.4f}").set_caption("Test Set Performance"))
    
    # Bar chart comparison
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    df.plot(kind='bar', ax=ax, colormap='viridis', edgecolor='white', width=0.7)
    ax.set_title('MST-GNN Performance: CSI500 vs VN100', fontsize=14, fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig("checkpoints/comparison_B.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No results found yet — run Steps 4 and 5 first.")

---
## Step 7 — Force Reset (Optional)

Run this cell **only** if you want to restart training from scratch.

In [ ]:
# ⚠️ UNCOMMENT TO RESET — deletes all checkpoints!
# import glob, os
# for f in glob.glob("checkpoints/ckpt_*.pt"):
#     os.remove(f)
#     print(f"Removed: {f}")
# if os.path.exists("checkpoints/best_model.pt"):
#     os.remove("checkpoints/best_model.pt")
#     print("Removed: checkpoints/best_model.pt")
# print("\n✅ All checkpoints removed. Re-run Steps 4-5 to train from scratch.")

---
## Dataset Status

In [ ]:
print("\n=== Dataset Status ===")
for ds, path_pattern in [
    ("CSI500", "data/processed/csi500_snapshots.pkl"),
    ("VN100",  "data/processed_vn/vn100_snapshots.pkl"),
]:
    if os.path.exists(path_pattern):
        import pickle
        with open(path_pattern, 'rb') as f:
            snaps = pickle.load(f)
        n = len(snaps)
        n_train = int(n * 0.7)
        n_val = int(n * 0.1)
        n_test = n - n_train - n_val
        print(f"{ds:8s}: {n} snapshots (train~{n_train}, val~{n_val}, test~{n_test})")
    else:
        print(f"{ds:8s}: not built yet")

print("\n=== Graph Cache ===")
for f in sorted(glob.glob("data/processed*/graphs_*.pkl")):
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")
if not glob.glob("data/processed*/graphs_*.pkl"):
    print("  (none yet — will be created during Phase 3)")

print("\n=== Checkpoints ===")
for f in sorted(glob.glob("checkpoints/*.pt")):
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")
if not glob.glob("checkpoints/*.pt"):
    print("  (none yet)")